# TSTL Phase R1 — Colab (GPU)

**Runtime → Restart session のあと、セル1から順に実行**（セル3だけ実行しない）。

In [ ]:
# セル1: clone/update + sys.path + pip
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/blabo25226/NSNandTSTL.git"
BRANCH = "20260713_create_TSTL"
CLONE_DIR = Path("/content/NSNandTSTL")
ROOT = CLONE_DIR / "TSTL"
SRC = ROOT / "src"


def run(cmd, cwd=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


def has_llm_freeze() -> bool:
    return (SRC / "llm_freeze.py").is_file()


if CLONE_DIR.is_dir() and not has_llm_freeze():
    print("Stale clone without llm_freeze — removing and re-cloning")
    shutil.rmtree(CLONE_DIR)

if not CLONE_DIR.is_dir():
    run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(CLONE_DIR)])
elif (CLONE_DIR / ".git").is_dir():
    run(["git", "fetch", "origin", BRANCH], cwd=CLONE_DIR)
    run(["git", "checkout", BRANCH], cwd=CLONE_DIR)
    run(["git", "pull", "origin", BRANCH], cwd=CLONE_DIR)

if not has_llm_freeze():
    raise FileNotFoundError(
        f"{SRC / 'llm_freeze.py'} missing. Push branch {BRANCH} to GitHub."
    )

src_str = str(SRC.resolve())
if src_str not in sys.path:
    sys.path.insert(0, src_str)
os.chdir(ROOT)

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-r.txt")])

print("ROOT:", ROOT)
print("llm_freeze:", has_llm_freeze())
print("sys.path[0]:", sys.path[0])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

CKPT_ROOT = Path('/content/drive/MyDrive/TSTL/r1')
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
print('checkpoint root:', CKPT_ROOT)

In [ ]:
import sys
from pathlib import Path

SRC = Path("/content/NSNandTSTL/TSTL/src")
if not (SRC / "llm_freeze.py").is_file():
    raise RuntimeError("セル1を先に実行してください")
src_str = str(SRC.resolve())
if src_str not in sys.path:
    sys.path.insert(0, src_str)

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llm_freeze import freeze_all_except_layers, num_transformer_layers
from llm_grpo import GrpoRunConfig, run_grpo_train
from llm_eval import accuracy_score
from llm_profile import default_r1_out_dir, profile_layers_from_scores, resume_layer_scan

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
TRAIN_N = 256
EVAL_N = 64
STEPS = 200
LR = 1e-5
SEED = 42
print('torch cuda:', torch.cuda.is_available())
print('llm_freeze:', __import__('llm_freeze').__file__)

In [ ]:
def load_gsm8k_prompts(n_train: int, n_eval: int):
    ds = load_dataset('gsm8k', 'main')
    train = ds['train'].select(range(n_train))
    test = ds['test'].select(range(n_eval))
    return train, test


def to_grpo_rows(split, tokenizer):
    rows = []
    for ex in split:
        prompt = f"Question: {ex['question']}\nAnswer:"
        rows.append({"prompt": prompt, "answer": ex['answer']})
    return rows


train_split, eval_split = load_gsm8k_prompts(TRAIN_N, EVAL_N)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
grpo_train = to_grpo_rows(train_split, tokenizer)
print('train rows:', len(grpo_train))

In [ ]:
def eval_model(model, split, tokenizer, max_new_tokens=64):
    preds, gold = [], []
    model.eval()
    for ex in split:
        prompt = f"Question: {ex['question']}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        text = tokenizer.decode(out[0], skip_special_tokens=True)
        preds.append(text)
        gold.append(ex['answer'])
    return accuracy_score(preds, gold)


base_model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map='auto')
s_base = eval_model(base_model, eval_split, tokenizer)
print('S_base', s_base)

In [ ]:
out_dir = CKPT_ROOT / 'scan_latest'
out_dir.mkdir(parents=True, exist_ok=True)

full_dir = out_dir / 'full'
full_model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map='auto')
run_grpo_train(
    full_model,
    grpo_train,
    GrpoRunConfig(output_dir=full_dir, learning_rate=LR, num_train_steps=STEPS, train_layer_indices=None),
)
s_full = eval_model(full_model, eval_split, tokenizer)
print('S_full', s_full)

In [ ]:
n_layers = num_transformer_layers(full_model)
print('num layers:', n_layers)


def train_and_eval_layer(k: int) -> float:
    layer_dir = out_dir / f'layer_{k}'
    m = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map='auto')
    run_grpo_train(
        m,
        grpo_train,
        GrpoRunConfig(
            output_dir=layer_dir,
            learning_rate=LR,
            num_train_steps=STEPS,
            train_layer_indices=[k],
        ),
    )
    return eval_model(m, eval_split, tokenizer)


result = resume_layer_scan(
    out_dir,
    n_layers,
    train_and_eval_layer,
    s_base=s_base,
    s_full=s_full,
    config={'model': MODEL, 'steps': STEPS, 'lr': LR, 'seed': SEED},
)
print(result.out_dir)
print('best C', max(result.contributions.values()))